In [ ]:
import time
START = time.time()

from concurrent.futures import ThreadPoolExecutor
import glob
import librosa
import numpy as np
import os
import pandas as pd
import re
import sys
import tensorflow as tf

tf.experimental.numpy.experimental_enable_numpy_behavior()

TERMINATE_TIME = START + 5300

In [ ]:
birdclassifier = tf.saved_model.load('/kaggle/input/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8/')

# Get the map from common bird name to dataframe index

In [ ]:
primary_labels = pd.read_csv('/kaggle/input/birdclef-2025/sample_submission.csv').columns[1:].to_list()
primary_labels_indices = range(len(primary_labels))

primary_labels_map = dict(zip(primary_labels, primary_labels_indices))

taxonomy = pd.read_csv('/kaggle/input/birdclef-2025/taxonomy.csv', index_col='common_name')['primary_label']
taxonomy_map = taxonomy.map(primary_labels_map)

common_names = taxonomy.index.to_list()

In [ ]:
bc_labels = pd.read_csv('/kaggle/input/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8/assets/label.csv').iloc[:, 0].to_list()
bc_labels_indices = range(len(bc_labels))

primary_labels_map = dict(zip(bc_labels, bc_labels_indices))

birdclassifier_last = len(bc_labels)
birdclassifier_indices = [primary_labels_map[pl] if pl in bc_labels else birdclassifier_last for pl in primary_labels]

In [ ]:
total_predicted_species = sum([pl in bc_labels for pl in primary_labels])
print(f'Note: we can predict {total_predicted_species} species only!')

# Get all the data files

In [ ]:
def get_oggs(max_oggs=8):
    if len(glob.glob('/kaggle/input/birdclef-2025/test_soundscapes/*.ogg')) > 0:
        oggs = glob.glob('/kaggle/input/birdclef-2025/test_soundscapes/*.ogg')
    else:
        oggs = sorted(glob.glob(f'/kaggle/input/birdclef-2025/train_soundscapes/*.ogg'))[:max_oggs]
    return [(n, ogg, re.search(r'/([^/]+)\.ogg$', ogg).group(1)) for n, ogg in enumerate(oggs)]

oggs = get_oggs()

# Process the files in threads

In [ ]:
def bvc_result(ogg):
    _, fname, ss_id = ogg
    sr = 32_000

    print(f'{ss_id}')
    row_ids = [f'{ss_id}_{n}' for n in range(5, 65, 5)]

    if time.time() > TERMINATE_TIME:
        return row_ids, -1000 * np.ones((12, 206))
        
    data, _ = librosa.load(fname, sr=sr)

    model_outputs = birdclassifier.infer_tf(data.reshape((-1, 5 * sr)))['label']
    model_outputs = tf.pad(model_outputs, tf.constant([[0, 0], [0, 1]]))
    result = model_outputs[:, birdclassifier_indices]

    return row_ids, result

In [ ]:
row_ids = []
result = []

with ThreadPoolExecutor(max_workers=4) as executor:
    for ogg_row_ids, ogg_result in executor.map(bvc_result, oggs):
        row_ids += ogg_row_ids
        result.append(ogg_result)

In [ ]:
submission = pd.DataFrame(np.concatenate(result), columns=primary_labels)
submission['row_id'] = row_ids
submission = submission[['row_id'] + primary_labels]

In [ ]:
# Write CSV
submission.to_csv('submission.csv', index=False)

# Display submission DataFrame
display(submission.head(20))

display(submission.tail(20))